In [1]:
# NAME: Project Two Dashboard
# DESCRIPTION: Script to create a Jupyter Dash dashboard for a database
# AUTHOR: Joshua Pickle
# LAST EDITED: 1/26/2026

# Used ChatGPT for some bug-fixing. Code is my own.

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the imports necessary for Flask-Login for security purposes
import flask
from flask_login import UserMixin, current_user, login_user, LoginManager, login_required, logout_user

# Import user manager
from Users_Python_Module import UserManager

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, callback, ctx, no_update
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure the plotting routines
import pandas as pd

# Allow access to Flask server (for Flask-Login)
server = flask.Flask(__name__)

# Import animal shelter
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Use the username and password and CRUD Python module name
username = 'aacuser'
password = 'CS340pwd'

# Connect to users database via User Manager
users = UserManager(username, password)

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Class read method must support return of list object and accept projection json input
# Sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

# Get Grazioso Salvare's logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#########################
# Configure Pages
#########################

# Set up app configuration, suppress callback exceptions to prevent annoying error
# (per the suggestion of the error)
app = JupyterDash(__name__, server = server, suppress_callback_exceptions = True)

# Show the displayed page
app.layout = html.Div([
    dcc.Location(id='url'),
    html.Div(id='page-content'),

    # Remember the number of times the "Logout" button has been pressed
    dcc.Store(id='logout-clicks', data = 1)
])

# Determine which page should be displayed based on url path
@callback(
    Output('page-content', 'children'), 
    Input('url', 'pathname')
)
def display_page(pathname):
    if pathname == '/database' and current_user.is_authenticated == True:
        return database_layout
    else:
        return login_page
        
#########################
# Login Setup
#########################

# Create login manager to enable Flask-Login
login = LoginManager(server)

# Create secret key for login_user
server.config['SECRET_KEY'] = 'samplekey'

# User class to implement UserMixin for Flask-Login
class User(UserMixin):
    # Function to create a user
    def __init__(self, username):
        self.id = username

    # Function to get the id from a user
    def get_id(self):
        return self.id

#########################
# Login Layout / View
#########################

# Login page
login_page = html.Div([
    html.Div(
        className='col s12 m6',
        children = [
            html.Center(html.Img(
                src = 'data:image/png;base64,{}'.format(encoded_image.decode()), 
                style = {
                    'height': '10%', 
                    'width': '10%'
                }
            ))
        ]
    ),
    html.Center(html.B(html.H1('Database Login'))),
    html.Center(html.B(html.H2('Username'))),
    html.Center(dcc.Input(
        id = 'username-input', 
        placeholder = 'Input Username', 
        value = ''
    )),
    html.Center(html.B(html.H3(
        'Username Cannot Be Empty', 
        id = 'bad-username-text', 
        style={'color': 'red'}, 
        hidden = True
    ))),
    html.Center(html.B(html.H2('Password'))),
    html.Center(dcc.Input(
        id = 'password-input', 
        type = 'password', 
        placeholder = 'Input Password', 
        value = ''
    )),
    html.Center(html.B(html.H3(
        'Password Cannot Be Empty', 
        id = 'bad-password-text', 
        style = {'color': 'red'}, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'Incorrect username/password combination', 
        id = 'incorrect-signin-text', 
        style = {'color': 'red'}, 
        hidden = True
    ))),
    html.Center(html.B(html.H3(
        'User with username does already exists', 
        id = 'user-existence-text', 
        style = {'color': 'red'}, 
        hidden = True
    ))),
    html.Div(
        # Make child Divs go next to each other
        className='row',
        style = {
            'display': 'flex', 
            'justifyContent': 'center', 
            'marginTop': 25
        },
        children = [   
            html.Button(
                id = 'login-button', 
                children = 'Log In', 
                style = {'backgroundColor': '#67bed9'},
                n_clicks = 0
            ),
            html.Button(
                id = 'register-button', 
                children = 'Register',
                style = {'backgroundColor': '#d07597', 'marginLeft': 15},
                n_clicks = 0
            )
        ]
    )
])

#############################################
# Interaction Between Login Components / Controller
#############################################

# Set login page when logged out
login.login_view = login_page

# Implement ability to get current user
@login.user_loader
def load_user(id):
    if users.doesUserExist(id) == True:
        return User(id)
    else:
        return None

# Function to attempt to access the database (via login or register)
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('incorrect-signin-text', 'hidden'),
    Output('user-existence-text', 'hidden'),
    Input('login-button', 'n_clicks'),
    Input('register-button', 'n_clicks'),
    State('username-input', 'value'),
    State('password-input', 'value'),
    prevent_initial_call = True
)
def attempt_database_access(login_clicks, register_clicks, username, password):
    # Make the extension "/login" immediately
    if (login_clicks == 0 and register_clicks == 0):
        return '/login', True, True
    # Prevent access if username or password are empty
    elif username == '' or password == '':
        return no_update, True, True
    # Login
    if 'login-button' == ctx.triggered_id:
        # Check if username/password combination exists
        if users.isSignInCorrect(username, password):
            user = User(username)
            login_user(user)
            return '/database', True, True
        else:
            return no_update, False, True
    # Registration
    elif 'register-button' == ctx.triggered_id:
        # Check if user exists
        if not users.doesUserExist(username):
            # Create user
            users.createUser(username, password)
            user = User(username)
            login_user(user)
            return '/database', True, True
        # User already exists
        else:
            return no_update, True, False
    # Neither button was pressed, do nothing
    return no_update, no_update, no_update

# Function to check if the username and password text boxes have contents for the error texts
@callback(
    Output('bad-username-text', 'hidden'),
    Output('bad-password-text', 'hidden'),
    Input('login-button', 'n_clicks'),
    Input('register-button', 'n_clicks'),
    State('username-input', 'value'),
    State('password-input', 'value'),
    prevent_initial_call = True
)
def user_pass_texts_full(login_clicks, register_clicks, username, password):
    usernameFull = True
    passwordFull = True
    
    if username == '':
        usernameFull = False
    if password == '':
        passwordFull = False
        
    return usernameFull, passwordFull

#########################
# Dashboard Layout / View
#########################

# Database page
database_layout = html.Div([
    html.Div(
        # Make child Divs go next to each other
        className='row',
        style={'display' : 'flex', 'height': '125px'},
        children = [
            # Display logo
            html.Div(
                className='col s12 m6',
                children = [
                    html.Img(
                        src = 'data:image/png;base64,{}'.format(encoded_image.decode()), 
                        style = {
                            'height': '100%', 
                            'width': '100%'
                        }
                    )
                ]
            ),
            # Show name text
            html.Div(
                className='col s12 m6',
                children = [
                    html.B(html.H1('CS-340 Dashboard', style = {'marginLeft': 550})),
                    html.B(html.H1('Written by Joshua Pickle', style = {'marginLeft': 500}))
                ]
            ),
            # Create logout button
            html.Div(
                className='col s12 m6',
                children = [
                    html.Button(
                        id = 'logout-button', 
                        children = 'Log Out',
                        style = {'backgroundColor': '#d07597', 'marginLeft': 450, 'marginTop': 50},
                        n_clicks = 0
                    )
                ]
            )
        ],
    ),
    html.Hr(),
    html.Div(
        # Make child Divs go next to each other
        className='row',
        style={'display' : 'flex'},
        children = [
            # Create radio buttons for choosing queries
            html.Div(
                className='col s12 m6',
                children = [
                    dcc.RadioItems(
                        id = 'filter-type',
                        options=[
                           {'label': 'Water Rescue', 'value': 'WaterRescue'},
                           {'label': 'Mountain Rescue', 'value': 'MtnRescue'},
                           {'label': 'Disaster Rescue', 'value': 'DisasterRescue'},
                           {'label': 'Reset', 'value': 'Default'},
                       ],
                       value='Default'
                    )
                ]
            ),
            # Create helper texts for explaining radio button functions
            html.Div(
                className='col s12 m6',
                children = [
                    html.U(html.P(
                        '?', 
                        title = 'Narrows search down to dogs fit for water rescue. Includes the following criteria:\nBreed is Laborator Retriever Mix, Chesapeake Bay Retriever, or Newfoundland\nSex upon outcome must be \"Intact Female\"\nAge upon outcome in weeks must be between 26 and 156', 
                        style = {
                            'marginTop': 0, 
                            'marginBottom': 0, 
                            'marginLeft': -23, 
                            'color': 'blue'
                        }
                    )),
                    html.U(html.P(
                        '?',
                        title = 'Narrows search down to dogs fit for mountain rescue. Includes the following criteria:\nBreed is German Shepherd, Alaskan Malamute, Old English Sheepdog, Siberian Husky, or Rottweiler\nSex upon outcome must be \"Intact Male\"\nAge upon outcome in weeks must be between 26 and 156',
                        style = {
                            'marginTop': 1, 
                            'marginBottom': 0, 
                            'marginLeft': 2, 
                            'color': 'blue'
                        }
                    )),
                    html.U(html.P(
                        '?',
                        title = 'Narrows search down to dogs fit for disaster rescue. Includes the following criteria:\nBreed is Doberman Pinscher, German Shepherd, Golden Retriever, Bloodhound, or Rottweiler\nSex upon outcome must be \"Intact Male\"\nAge upon outcome in weeks must be between 20 and 300',
                        style = {
                            'marginTop': 1, 
                            'marginBottom': 0, 
                            'marginLeft': -8, 
                            'color': 'blue'
                        }
                    )),
                    html.U(html.P(
                        '?', 
                        title = 'Resets the applied category, returning results to show all dogs (aside from categories applied on the table itself)',
                        style = {
                            'marginTop': 2, 
                            'marginBottom': 0, 
                            'marginLeft': -75, 
                            'color': 'blue'
                        }
                    ))
                ]
            )
        ]
    ), 
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[],
        page_action="native",
        page_current=0,
        page_size=10
    ),
    html.Br(),
    html.Hr(),
    # This sets up the dashboard so that the chart and the geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display' : 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                children=[
                    dl.Map(
                        center=[56, 10], 
                        zoom=6, 
                        style={"height": "50vh"}, 
                        children=[
                            dl.TileLayer(id="base-layer-id")
                        ]
                    )
                ]
            )
        ]
    )
])

#############################################
# Interaction Between Database Components / Controller
#############################################

@app.callback(Output('datatable-id','data'),
            Output('datatable-id', 'columns'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if (filter_type == "WaterRescue"):
        readQuery = dict()
        readQuery = {'animal_type': 'Dog', 'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
                    'sex_upon_outcome': 'Intact Female', 'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
        data = FixQueryResultIds(list(db.read(readQuery)))
    elif (filter_type == "MtnRescue"):
        readQuery = dict()
        readQuery = {'animal_type': 'Dog', 'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']},
                    'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
        data = FixQueryResultIds(list(db.read(readQuery)))
    elif (filter_type == "DisasterRescue"):
        readQuery = dict()
        readQuery = {'animal_type': 'Dog', 'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']},
                    'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}}
        data = FixQueryResultIds(list(db.read(readQuery)))
    else: 
        data=df.to_dict('records')
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
        
    return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
    Input('filter-type', 'value')])
def update_graphs(viewData, filter_type):
    # Display pie chart
    dff = pd.DataFrame.from_dict(viewData)
    
    return [
        dcc.Graph(            
            figure = px.pie(dff, names='breed', title='Preferred Animals')
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if viewData is None:
        return
    elif index == [] or index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)

    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else:
        row = index[0]

    # Variable to adjust for displacement caused by _id column randomly appearing
    if type(dff.iloc[row, 6]) is str:
        columnMod = 1
    else:
        columnMod = 0
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '800px', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id = "base-layer-id"),
            # Marker with tool tip and popup
            # Columns 6 and 7 define the grid-coordinates for the map
            # Column 2 defines the breed for the animal
            # Column 5 defines the name of the animal
            dl.Marker(position=[dff.iloc[row, 6 + columnMod],dff.iloc[row, 7 + columnMod]], children=[
                dl.Tooltip(dff.iloc[row, 2 + columnMod]),
                dl.Popup([
                    html.P("Animal Name:"),
                    html.H1(dff.iloc[row, 5 + columnMod])
                ])
            ])
        ])
    ]

# Function to log the user out
@callback(
    Output('url', 'pathname', allow_duplicate = True),
    Output('logout-clicks', 'data'),
    Input('logout-button', 'n_clicks'),
    State('logout-clicks', 'data'),
    prevent_initial_call = True
)
def log_user_out(logout_clicks, stored_clicks):
    if logout_clicks is None or logout_clicks < stored_clicks:
        return no_update, stored_clicks

    # Only logout on an actual click
    logout_user()
    
    return '/login', stored_clicks + 1

# Function to set the _id of the results of a query to strings
def FixQueryResultIds(readQuery):
    for result in readQuery:
        if '_id' in result:
            result['_id'] = str(result['_id'])
    return readQuery

# Run app and display result in jupyterlab mode
# If you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()

Dash is running on http://127.0.0.1:8050/

Dash app running on http://127.0.0.1:8050/
